# 🧠 Multi-Head Attention
**Course:** Deep Learning  
**Lecture:** 06 — Attention Mechanisms and Transformers  
**Part:** b — Multi-Head Attention

---

> **Where we are:** We know how single attention works. Now, we expand it so the model can focus on different aspects of a sequence simultaneously.

**What you'll learn:**
- Multi-Head Attention equations
- Implementing heads in parallel
- Projection matrices $W^Q, W^K, W^V, W^O$

**Exam relevance:** ⭐⭐⭐⭐ (Parameter counting for MHA is a classic exam question)


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns

# Confirm device
print(f"TensorFlow {tf.__version__} | Devices: {[d.name for d in tf.config.list_physical_devices()]}")

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12


## 📖 Theory: Why Multiple Heads?

$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O$

where $\text{head}_i = \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V)$

**Why multiple heads?**
- A single attention head aggregates all information based on one similarity metric.
- With multiple heads, Head 1 might learn syntactic dependencies (e.g., verbs to subjects), while Head 2 learns semantic dependencies (e.g., pronouns to entities).
- The dimension $d_{model}$ is typically divided by $h$ to get $d_k$ (e.g. 512 / 8 = 64). Thus, the computation cost is roughly the same as one large attention head.


## 🔀 Your Options

At this step, you can choose between:
- **Sequential Heads**: Iterate through heads (slow).
- **Parallel Heads (Reshape)**: Treat heads as a dimension in the tensor and compute all at once via matrix multiplication (fast, used in practice).


In [ ]:
class CustomMultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, 'd_model must be divisible by num_heads'
        self.num_heads = num_heads
        self.d_model = d_model
        self.d_k = d_model // num_heads   # dimension per head

        # Projection matrices for Q, K, V and the output
        self.Wq = tf.keras.layers.Dense(d_model, use_bias=False)
        self.Wk = tf.keras.layers.Dense(d_model, use_bias=False)
        self.Wv = tf.keras.layers.Dense(d_model, use_bias=False)
        self.Wo = tf.keras.layers.Dense(d_model, use_bias=False)  # output projection

    def split_heads(self, x, batch_size):
        # Reshape (batch, seq, d_model) -> (batch, num_heads, seq, d_k)
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.d_k))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, x):
        batch_size = tf.shape(x)[0]

        Q = self.split_heads(self.Wq(x), batch_size)  # (batch, heads, seq, d_k)
        K = self.split_heads(self.Wk(x), batch_size)
        V = self.split_heads(self.Wv(x), batch_size)

        # Scaled dot-product attention per head
        d_k = tf.cast(self.d_k, tf.float32)
        scores = tf.matmul(Q, K, transpose_b=True) / tf.sqrt(d_k)  # (batch, heads, seq, seq)
        weights = tf.nn.softmax(scores, axis=-1)
        attended = tf.matmul(weights, V)                             # (batch, heads, seq, d_k)

        # Concatenate heads: (batch, heads, seq, d_k) -> (batch, seq, d_model)
        attended = tf.transpose(attended, perm=[0, 2, 1, 3])
        concat = tf.reshape(attended, (batch_size, -1, self.d_model))

        output = self.Wo(concat)  # final linear projection
        return output, weights

# Test and compare against Keras built-in
d_model, num_heads, seq_len = 64, 4, 10
x_sample = tf.random.normal((2, seq_len, d_model))

custom_mha = CustomMultiHeadAttention(d_model, num_heads)
out_custom, attn_weights = custom_mha(x_sample)
print(f'Custom MHA output: {out_custom.shape}')  # (2, 10, 64)
print(f'Attention weights: {attn_weights.shape}') # (2, 4, 10, 10)

# Visualise each head\'s attention pattern
fig, axes = plt.subplots(1, num_heads, figsize=(16, 3))
for h in range(num_heads):
    axes[h].imshow(attn_weights[0, h].numpy(), cmap='Blues', vmin=0, vmax=1)
    axes[h].set_title(f'Head {h+1}')
    axes[h].set_xlabel('Key pos'); axes[h].set_ylabel('Query pos')
plt.suptitle('Multi-head attention weights - each head attends differently')
plt.tight_layout(); plt.show()

In [ ]:
class CustomMultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, 'd_model must be divisible by num_heads'
        self.num_heads = num_heads
        self.d_model = d_model
        self.d_k = d_model // num_heads   # dimension per head

        # Projection matrices for Q, K, V and the output
        self.Wq = tf.keras.layers.Dense(d_model, use_bias=False)
        self.Wk = tf.keras.layers.Dense(d_model, use_bias=False)
        self.Wv = tf.keras.layers.Dense(d_model, use_bias=False)
        self.Wo = tf.keras.layers.Dense(d_model, use_bias=False)  # output projection

    def split_heads(self, x, batch_size):
        # Reshape (batch, seq, d_model) -> (batch, num_heads, seq, d_k)
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.d_k))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, x):
        batch_size = tf.shape(x)[0]

        Q = self.split_heads(self.Wq(x), batch_size)  # (batch, heads, seq, d_k)
        K = self.split_heads(self.Wk(x), batch_size)
        V = self.split_heads(self.Wv(x), batch_size)

        # Scaled dot-product attention per head
        d_k = tf.cast(self.d_k, tf.float32)
        scores = tf.matmul(Q, K, transpose_b=True) / tf.sqrt(d_k)  # (batch, heads, seq, seq)
        weights = tf.nn.softmax(scores, axis=-1)
        attended = tf.matmul(weights, V)                             # (batch, heads, seq, d_k)

        # Concatenate heads: (batch, heads, seq, d_k) -> (batch, seq, d_model)
        attended = tf.transpose(attended, perm=[0, 2, 1, 3])
        concat = tf.reshape(attended, (batch_size, -1, self.d_model))

        output = self.Wo(concat)  # final linear projection
        return output, weights

# Test and compare against Keras built-in
d_model, num_heads, seq_len = 64, 4, 10
x_sample = tf.random.normal((2, seq_len, d_model))

custom_mha = CustomMultiHeadAttention(d_model, num_heads)
out_custom, attn_weights = custom_mha(x_sample)
print(f'Custom MHA output: {out_custom.shape}')  # (2, 10, 64)
print(f'Attention weights: {attn_weights.shape}') # (2, 4, 10, 10)

# Visualise each head\'s attention pattern
fig, axes = plt.subplots(1, num_heads, figsize=(16, 3))
for h in range(num_heads):
    axes[h].imshow(attn_weights[0, h].numpy(), cmap='Blues', vmin=0, vmax=1)
    axes[h].set_title(f'Head {h+1}')
    axes[h].set_xlabel('Key pos'); axes[h].set_ylabel('Query pos')
plt.suptitle('Multi-head attention weights - each head attends differently')
plt.tight_layout(); plt.show()

In [ ]:
# 🔬 Comparison: Visualize Attention Heads

# Let's generate a random set of weights simulating what different heads might look like
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for i in range(4):
    ax = axes[i//2, i%2]
    # In a real model, this would be weights[0, i]
    # Here we show dummy patterns: Head 0 might look at next word, Head 1 at previous, etc.
    sns.heatmap(tf.random.uniform((10, 10)), ax=ax, cmap='Blues', cbar=False)
    ax.set_title(f"Head {i+1}")
plt.tight_layout()
plt.show()

# Parameter count comparison:
# Wq, Wk, Wv, Wo all take (d_model, d_model) parameters
print(f"Total params for d_model=512: {4 * 512 * 512}")


## 🎓 Exam Corner

### Question 1 (Conceptual)
If `d_model=512` and `num_heads=8`, what is `d_k` for each head? What is the total parameter count of a MultiHeadAttention layer?

<details><summary>💡 Answer</summary>

- `d_k` = 512 / 8 = 64.
- Total parameters = 4 weight matrices ($W^Q, W^K, W^V, W^O$).
  Each is `d_model` x `d_model` (plus biases, usually).
  Roughly $4 \times 512^2 = 1,048,576$ parameters (excluding biases).

</details>

### Question 2 (Practical)
Why is Multi-Head Attention better than a single large attention? What is the $W^O$ matrix for and what is its shape?

<details><summary>💡 Answer</summary>

- Multiple heads allow the model to jointly attend to information from different representation subspaces at different positions. A single head averages this out.
- $W^O$ (Output weight matrix) mixes the concatenated outputs from all the different heads back into the original $d_{model}$ dimension. Its shape is `(d_model, d_model)`.

</details>
